In [38]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from numpy import pi

In [39]:
def build_cuccaro_circuit(a, b):
    qreg_q = QuantumRegister(10, 'q')
    creg_c = ClassicalRegister(5, 'c')
    circuit = QuantumCircuit(qreg_q, creg_c)

    # q1,q4,q7,q10 = bits of A (LSB to MSB)
    # q2,q5,q8,q11 = bits of B (LSB to MSB)

    # encode A
    if a & 1: circuit.x(qreg_q[1])
    if a & 2: circuit.x(qreg_q[4])
    if a & 4: circuit.x(qreg_q[6])
    if a & 8: circuit.x(qreg_q[8])

    # encode B
    if b & 1: circuit.x(qreg_q[0])
    if b & 2: circuit.x(qreg_q[3])
    if b & 4: circuit.x(qreg_q[5])
    if b & 8: circuit.x(qreg_q[7])

    circuit.barrier()

    circuit.cx(qreg_q[6], qreg_q[5])
    circuit.cx(qreg_q[4], qreg_q[3])
    circuit.cx(qreg_q[8], qreg_q[7])
    circuit.cx(qreg_q[4], qreg_q[2])
    circuit.ccx(qreg_q[0], qreg_q[1], qreg_q[2])
    circuit.cx(qreg_q[6], qreg_q[4])
    circuit.ccx(qreg_q[2], qreg_q[3], qreg_q[4])
    circuit.cx(qreg_q[8], qreg_q[6])
    circuit.ccx(qreg_q[4], qreg_q[5], qreg_q[6])
    circuit.cx(qreg_q[8], qreg_q[9])
    circuit.barrier(qreg_q[3])
    circuit.x(qreg_q[3])
    circuit.ccx(qreg_q[6], qreg_q[7], qreg_q[9])
    circuit.x(qreg_q[5])
    circuit.cx(qreg_q[2], qreg_q[3])
    circuit.cx(qreg_q[4], qreg_q[5])
    circuit.cx(qreg_q[6], qreg_q[7])
    circuit.ccx(qreg_q[4], qreg_q[5], qreg_q[6])
    circuit.ccx(qreg_q[2], qreg_q[3], qreg_q[4])
    circuit.x(qreg_q[5])
    circuit.cx(qreg_q[8], qreg_q[6])
    circuit.ccx(qreg_q[0], qreg_q[1], qreg_q[2])
    circuit.x(qreg_q[3])
    circuit.cx(qreg_q[6], qreg_q[4])
    circuit.barrier(qreg_q[8])
    circuit.cx(qreg_q[4], qreg_q[2])
    circuit.barrier(qreg_q[0])
    circuit.barrier(qreg_q[6])
    circuit.barrier(qreg_q[8])
    circuit.cx(qreg_q[6], qreg_q[5])
    circuit.cx(qreg_q[8], qreg_q[7])
    circuit.cx(qreg_q[1], qreg_q[0])
    circuit.cx(qreg_q[4], qreg_q[3])
    circuit.measure(qreg_q[0], creg_c[0])
    circuit.measure(qreg_q[3], creg_c[1])
    circuit.measure(qreg_q[5], creg_c[2])
    circuit.measure(qreg_q[7], creg_c[3])
    circuit.measure(qreg_q[9], creg_c[4])


    return circuit

In [40]:
import numpy as np

shots = 1000

# ====================================================
# BITFLIP NOISE MODEL
# ====================================================

p = 0.0337

# ----------------------------------------------------
# 1-Qubit Bitflip Error
# ----------------------------------------------------

error_1 = pauli_error([
    ('X', p),
    ('I', 1-p)
])

# ----------------------------------------------------
# 2-Qubit Error
# E ⊗ E
# ----------------------------------------------------

error_2 = error_1.tensor(error_1)

# ----------------------------------------------------
# 3-Qubit Error
# E ⊗ E ⊗ E
# ----------------------------------------------------

error_3 = error_1.tensor(error_1).tensor(error_1)

# ====================================================
# CREATE NOISE MODEL
# ====================================================

noise_model = NoiseModel()

# ----------------------------------------------------
# Apply noise to X gates
# ----------------------------------------------------

noise_model.add_all_qubit_quantum_error(
    error_1,
    ['x']
)

# ----------------------------------------------------
# Apply noise to CX gates
# ----------------------------------------------------

noise_model.add_all_qubit_quantum_error(
    error_2,
    ['cx']
)

# ----------------------------------------------------
# Apply noise to CCX gates
# ----------------------------------------------------

noise_model.add_all_qubit_quantum_error(
    error_3,
    ['ccx']
)

sim = AerSimulator(noise_model=noise_model)

# ====================================================
# NUMPY ARRAYS FOR METRICS
# ====================================================

ER_arr = np.zeros((16, 16))
NMED_arr = np.zeros((16, 16))
MRED_arr = np.zeros((16, 16))

for a in range(16):
    for b in range(16):

        qc = build_cuccaro_circuit(a, b)

        compiled = transpile(qc, sim)

        counts = sim.run(
            compiled,
            shots=shots
        ).result().get_counts()

        correct = a + b
        correct_output = format(correct, '05b')

        correct_counts = counts.get(correct_output, 0)

        ER = 1 - correct_counts / shots

        total_ED = 0
        total_relative_ED = 0

        for output, freq in counts.items():

            noisy_decimal = int(output, 2)

            ED = abs(correct - noisy_decimal)

            total_ED += ED * freq

            if correct != 0:
                total_relative_ED += (ED / correct) * freq

        mean_ED = total_ED / shots

        D = 30

        NMED = mean_ED / D
        MRED = total_relative_ED / shots

        ER_arr[a, b] = ER
        NMED_arr[a, b] = NMED
        MRED_arr[a, b] = MRED

# ====================================================
# OVERALL METRICS
# ====================================================

er = np.mean(ER_arr)
nmed = np.mean(NMED_arr)
mred = np.mean(MRED_arr)

print("Bitflip Noise\n")

print(f"ER (%) : {er * 100:.2f}")
print(f"NMED   : {nmed:.3f}")
print(f"MRED   : {mred:.3f}")

# Flatten to 1D if needed
er_array = ER_arr.flatten()

print("\nER Array:")
print(er_array)

Bitflip Noise

ER (%) : 77.81
NMED   : 0.159
MRED   : 0.472

ER Array:
[0.788 0.805 0.772 0.768 0.794 0.763 0.749 0.759 0.787 0.805 0.783 0.769
 0.791 0.785 0.742 0.77  0.767 0.779 0.762 0.777 0.762 0.742 0.764 0.759
 0.769 0.777 0.759 0.785 0.789 0.788 0.773 0.792 0.765 0.747 0.753 0.745
 0.753 0.742 0.763 0.781 0.73  0.765 0.775 0.787 0.755 0.765 0.78  0.782
 0.746 0.769 0.762 0.805 0.761 0.781 0.779 0.78  0.742 0.758 0.771 0.793
 0.746 0.787 0.785 0.79  0.756 0.789 0.756 0.745 0.77  0.78  0.773 0.752
 0.766 0.801 0.768 0.76  0.776 0.775 0.784 0.779 0.764 0.782 0.749 0.754
 0.808 0.792 0.801 0.802 0.764 0.783 0.775 0.794 0.799 0.8   0.787 0.791
 0.749 0.757 0.771 0.767 0.787 0.779 0.793 0.777 0.753 0.761 0.775 0.777
 0.773 0.762 0.802 0.784 0.727 0.797 0.764 0.798 0.794 0.781 0.787 0.799
 0.755 0.781 0.769 0.811 0.777 0.804 0.795 0.802 0.779 0.802 0.776 0.781
 0.788 0.771 0.782 0.75  0.805 0.827 0.79  0.771 0.803 0.784 0.775 0.779
 0.795 0.768 0.79  0.769 0.786 0.779 0.77  0.78  0.80